# LLM-7B: 7B Pretraining from Scratch on KaggleThis notebook contains the full pretraining system. Run all cells to train.- All model weights are **randomly initialized** (no pretrained weights).- Model: 7B params, GQA, RoPE, RMSNorm, SwiGLU.- Checkpoints auto-save and auto-resume across sessions.Instructions:1. Set **Accelerator = GPU T4x2 or P100**, **Internet = On** (for online mode)2. For **offline mode**: mount local data/tokenizer as Kaggle input datasets, update config3. Run all cells4. To resume in a new session, just re-run the training cell

In [ ]:
# Cell 1: Install dependencies!pip install -q --upgrade pip!pip install -q torch transformers datasets tokenizers accelerate pyyaml numpy tqdm pyarrow

In [ ]:
# Cell 2: Create project structureimport osfor d in ["configs", "src", "scripts"]:    os.makedirs(d, exist_ok=True)print("Project directories ready")

In [ ]:
%%writefile requirements.txttorch>=2.2.0transformers>=4.44.0datasets>=2.20.0tokenizers>=0.19.0accelerate>=0.33.0pyyaml>=6.0.1numpy>=1.26.0tqdm>=4.66.0

In [ ]:
%%writefile configs/pretrain_7b_kaggle.yamlrun_name: llm-7b-pretrain-v1seed: 42output_dir: /kaggle/working/checkpoints/llm-7bcache_dir: /kaggle/working/cachemodel:  vocab_size: 65536  hidden_size: 4096  intermediate_size: 13056  num_hidden_layers: 32  num_attention_heads: 32  num_key_value_heads: 8  max_position_embeddings: 4096  rope_theta: 500000.0  rms_norm_eps: 1.0e-5  tie_word_embeddings: false  use_flash_attn: true  gradient_checkpointing: truetokenizer:  pretrained_or_path: null  fallback_tokenizer: gpt2  train_if_missing: true  tokenizer_out_dir: /kaggle/working/tokenizer_65k  vocab_size: 65536  sample_chars_for_training: 20000000  # Offline mode: set these if you have local data  # local_data_dir: /kaggle/input/my-corpus  # directory with .jsonl/.parquet files  # local_fallback_path: /kaggle/input/my-tokenizer  # local tokenizer directorydata:  seq_len: 2048  num_workers: 2  shuffle_buffer: 10000  sources:    # Online mode (requires Internet=On):    - name: fineweb_edu      dataset_id: HuggingFaceFW/fineweb-edu      config_name: sample-10BT      split: train      text_field: text      weight: 0.45      streaming: true    - name: fineweb      dataset_id: HuggingFaceFW/fineweb      config_name: sample-10BT      split: train      text_field: text      weight: 0.25      streaming: true    - name: starcoder_code      dataset_id: bigcode/starcoderdata      config_name: null      data_dir: python      split: train      text_field: content      weight: 0.15      streaming: true    - name: open_web_math      dataset_id: open-web-math/open-web-math      config_name: null      split: train      text_field: text      weight: 0.10      streaming: true    - name: wikipedia_en      dataset_id: wikimedia/wikipedia      config_name: 20231101.en      split: train      text_field: text      weight: 0.05      streaming: true    # Offline mode (uncomment and mount local data as Kaggle input dataset):    # - name: my_corpus    #   local_dir: /kaggle/input/my-corpus    #   weight: 1.0training:  micro_batch_size: 1  gradient_accumulation_steps: 32  max_steps: 100000  warmup_steps: 1000  learning_rate: 3.0e-4  min_lr_ratio: 0.1  weight_decay: 0.1  adam_beta1: 0.9  adam_beta2: 0.95  adam_eps: 1.0e-8  max_grad_norm: 1.0  precision: bf16  log_interval: 10  save_interval: 1000  eval_interval: 1000  max_checkpoints_to_keep: 3  max_runtime_seconds: 39600  resume_from_checkpoint: autoeval:  eval_batches: 25  perplexity_prompts:    - "In statistical machine learning, cross-entropy loss measures the divergence between"    - "def binary_search(arr: list[int], target: int) -> int:"    - "To solve a system of linear equations using Gaussian elimination, we first"  completion_prompts:    - "The key difference between grouped-query attention and multi-head attention is"    - "A concise explanation of gradient checkpointing in deep neural networks:"

In [ ]:
%%writefile src/__init__.py"""7B Decoder-Only Transformer Pretraining Package."""__version__ = "0.1.0"

In [ ]:
%%writefile src/model.pyfrom dataclasses import dataclassimport mathfrom typing import Optional, Tupleimport torchimport torch.nn as nnimport torch.nn.functional as F@dataclassclass ModelConfig:    vocab_size: int = 65536    hidden_size: int = 4096    intermediate_size: int = 13056    num_hidden_layers: int = 32    num_attention_heads: int = 32    num_key_value_heads: int = 8    max_position_embeddings: int = 4096    rope_theta: float = 500000.0    rms_norm_eps: float = 1e-5    tie_word_embeddings: bool = False    use_flash_attn: bool = True    gradient_checkpointing: bool = True    @property    def head_dim(self) -> int:        assert self.hidden_size % self.num_attention_heads == 0        return self.hidden_size // self.num_attention_heads    @property    def num_key_value_groups(self) -> int:        assert self.num_attention_heads % self.num_key_value_heads == 0        return self.num_attention_heads // self.num_key_value_headsclass RMSNorm(nn.Module):    def __init__(self, dim: int, eps: float = 1e-5) -> None:        super().__init__()        self.eps = eps        self.weight = nn.Parameter(torch.ones(dim))    def forward(self, x: torch.Tensor) -> torch.Tensor:        orig_dtype = x.dtype        x_fp32 = x.float()        variance = x_fp32.pow(2).mean(dim=-1, keepdim=True)        normed = x_fp32 * torch.rsqrt(variance + self.eps)        return (normed * self.weight.float()).to(orig_dtype)class RotaryEmbedding(nn.Module):    def __init__(self, dim: int, max_seq_len: int = 4096, theta: float = 500000.0) -> None:        super().__init__()        self.dim = dim        self.max_seq_len = max_seq_len        self.theta = theta        inv_freq = 1.0 / (            self.theta ** (torch.arange(0, self.dim, 2, dtype=torch.float32) / self.dim)        )        self.register_buffer("inv_freq", inv_freq, persistent=False)        self._build_cos_sin(max_seq_len, device=inv_freq.device)    def _build_cos_sin(self, seq_len: int, device: torch.device) -> None:        t = torch.arange(seq_len, device=device, dtype=torch.float32)        freqs = torch.outer(t, self.inv_freq.to(device))        emb = torch.cat((freqs, freqs), dim=-1)        self.register_buffer("cos_cached", emb.cos(), persistent=False)        self.register_buffer("sin_cached", emb.sin(), persistent=False)        self._cached_len = seq_len    def forward(self, seq_len: int, device: torch.device, dtype: torch.dtype) -> Tuple[torch.Tensor, torch.Tensor]:        if seq_len > getattr(self, "_cached_len", 0) or self.cos_cached.device != device:            self._build_cos_sin(max(seq_len, self.max_seq_len), device=device)        cos = self.cos_cached[:seq_len].to(dtype=dtype)        sin = self.sin_cached[:seq_len].to(dtype=dtype)        return cos, sindef _rotate_half(x: torch.Tensor) -> torch.Tensor:    half = x.shape[-1] // 2    x1 = x[..., :half]    x2 = x[..., half:]    return torch.cat((-x2, x1), dim=-1)def apply_rotary_pos_emb(    q: torch.Tensor,    k: torch.Tensor,    cos: torch.Tensor,    sin: torch.Tensor,    position_ids: Optional[torch.Tensor] = None,) -> Tuple[torch.Tensor, torch.Tensor]:    if position_ids is not None:        cos_pos = cos[position_ids].unsqueeze(1)        sin_pos = sin[position_ids].unsqueeze(1)    else:        seq_len = q.shape[2]        cos_pos = cos[:seq_len].unsqueeze(0).unsqueeze(0)        sin_pos = sin[:seq_len].unsqueeze(0).unsqueeze(0)    q_embed = (q * cos_pos) + (_rotate_half(q) * sin_pos)    k_embed = (k * cos_pos) + (_rotate_half(k) * sin_pos)    return q_embed, k_embeddef repeat_kv(hidden_states: torch.Tensor, n_rep: int) -> torch.Tensor:    if n_rep == 1:        return hidden_states    batch, num_kv_heads, seq_len, head_dim = hidden_states.shape    expanded = hidden_states[:, :, None, :, :].expand(        batch, num_kv_heads, n_rep, seq_len, head_dim    )    return expanded.reshape(batch, num_kv_heads * n_rep, seq_len, head_dim)class GroupedQueryAttention(nn.Module):    def __init__(self, config: ModelConfig) -> None:        super().__init__()        self.config = config        self.hidden_size = config.hidden_size        self.num_heads = config.num_attention_heads        self.num_kv_heads = config.num_key_value_heads        self.num_kv_groups = config.num_key_value_groups        self.head_dim = config.head_dim        self.q_proj = nn.Linear(self.hidden_size, self.num_heads * self.head_dim, bias=False)        self.k_proj = nn.Linear(self.hidden_size, self.num_kv_heads * self.head_dim, bias=False)        self.v_proj = nn.Linear(self.hidden_size, self.num_kv_heads * self.head_dim, bias=False)        self.o_proj = nn.Linear(self.num_heads * self.head_dim, self.hidden_size, bias=False)    def forward(        self,        hidden_states: torch.Tensor,        cos: torch.Tensor,        sin: torch.Tensor,        position_ids: Optional[torch.Tensor] = None,        attention_mask: Optional[torch.Tensor] = None,    ) -> torch.Tensor:        bsz, q_len, _ = hidden_states.shape        q = self.q_proj(hidden_states).view(bsz, q_len, self.num_heads, self.head_dim).transpose(1, 2)        k = self.k_proj(hidden_states).view(bsz, q_len, self.num_kv_heads, self.head_dim).transpose(1, 2)        v = self.v_proj(hidden_states).view(bsz, q_len, self.num_kv_heads, self.head_dim).transpose(1, 2)        q, k = apply_rotary_pos_emb(q, k, cos, sin, position_ids=position_ids)        k = repeat_kv(k, self.num_kv_groups)        v = repeat_kv(v, self.num_kv_groups)        is_causal = attention_mask is None and q_len > 1        attn_output = F.scaled_dot_product_attention(            q,            k,            v,            attn_mask=attention_mask,            dropout_p=0.0,            is_causal=is_causal,        )        attn_output = attn_output.transpose(1, 2).contiguous().view(bsz, q_len, self.hidden_size)        return self.o_proj(attn_output)class SwiGLUFeedForward(nn.Module):    def __init__(self, config: ModelConfig) -> None:        super().__init__()        self.gate_proj = nn.Linear(config.hidden_size, config.intermediate_size, bias=False)        self.up_proj = nn.Linear(config.hidden_size, config.intermediate_size, bias=False)        self.down_proj = nn.Linear(config.intermediate_size, config.hidden_size, bias=False)    def forward(self, x: torch.Tensor) -> torch.Tensor:        return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))class TransformerBlock(nn.Module):    def __init__(self, config: ModelConfig) -> None:        super().__init__()        self.input_layernorm = RMSNorm(config.hidden_size, eps=config.rms_norm_eps)        self.self_attn = GroupedQueryAttention(config)        self.post_attention_layernorm = RMSNorm(config.hidden_size, eps=config.rms_norm_eps)        self.mlp = SwiGLUFeedForward(config)    def forward(        self,        hidden_states: torch.Tensor,        cos: torch.Tensor,        sin: torch.Tensor,        position_ids: Optional[torch.Tensor] = None,        attention_mask: Optional[torch.Tensor] = None,    ) -> torch.Tensor:        residual = hidden_states        hidden_states = self.input_layernorm(hidden_states)        hidden_states = self.self_attn(            hidden_states=hidden_states,            cos=cos,            sin=sin,            position_ids=position_ids,            attention_mask=attention_mask,        )        hidden_states = residual + hidden_states        residual = hidden_states        hidden_states = self.post_attention_layernorm(hidden_states)        hidden_states = residual + self.mlp(hidden_states)        return hidden_statesclass Llama7BForCausalLM(nn.Module):    def __init__(self, config: ModelConfig) -> None:        super().__init__()        self.config = config        self.embed_tokens = nn.Embedding(config.vocab_size, config.hidden_size)        self.rotary_emb = RotaryEmbedding(            dim=config.head_dim,            max_seq_len=config.max_position_embeddings,            theta=config.rope_theta,        )        self.layers = nn.ModuleList([TransformerBlock(config) for _ in range(config.num_hidden_layers)])        self.norm = RMSNorm(config.hidden_size, eps=config.rms_norm_eps)        if config.tie_word_embeddings:            self.lm_head = None        else:            self.lm_head = nn.Linear(config.hidden_size, config.vocab_size, bias=False)        self.apply(self._init_weights)        scale = 1.0 / math.sqrt(2.0 * config.num_hidden_layers)        for layer in self.layers:            nn.init.normal_(layer.self_attn.o_proj.weight, mean=0.0, std=0.02 * scale)            nn.init.normal_(layer.mlp.down_proj.weight, mean=0.0, std=0.02 * scale)    def _init_weights(self, module: nn.Module) -> None:        if isinstance(module, nn.Linear):            nn.init.normal_(module.weight, mean=0.0, std=0.02)            if module.bias is not None:                nn.init.zeros_(module.bias)        elif isinstance(module, nn.Embedding):            nn.init.normal_(module.weight, mean=0.0, std=0.02)    def count_parameters(self) -> dict[str, int]:        total = sum(p.numel() for p in self.parameters())        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)        embed = self.embed_tokens.weight.numel()        lm_head = 0 if self.lm_head is None else self.lm_head.weight.numel()        non_embed = total - embed - lm_head        return {            "total": total,            "trainable": trainable,            "non_embedding": non_embed,            "embeddings_and_head": embed + lm_head,        }    def forward(        self,        input_ids: torch.Tensor,        labels: Optional[torch.Tensor] = None,        position_ids: Optional[torch.Tensor] = None,        attention_mask: Optional[torch.Tensor] = None,    ) -> dict[str, Optional[torch.Tensor]]:        bsz, seq_len = input_ids.shape        hidden_states = self.embed_tokens(input_ids)        cos, sin = self.rotary_emb(seq_len=seq_len, device=hidden_states.device, dtype=hidden_states.dtype)        use_ckpt = self.config.gradient_checkpointing and self.training        for layer in self.layers:            if use_ckpt:                hidden_states = torch.utils.checkpoint.checkpoint(                    layer,                    hidden_states,                    cos,                    sin,                    position_ids,                    attention_mask,                    use_reentrant=False,                )            else:                hidden_states = layer(                    hidden_states=hidden_states,                    cos=cos,                    sin=sin,                    position_ids=position_ids,                    attention_mask=attention_mask,                )        hidden_states = self.norm(hidden_states)        logits = (            F.linear(hidden_states, self.embed_tokens.weight)            if self.lm_head is None            else self.lm_head(hidden_states)        )        loss = None        if labels is not None:            shift_logits = logits[..., :-1, :].contiguous()            shift_labels = labels[..., 1:].contiguous()            loss = F.cross_entropy(                shift_logits.view(-1, self.config.vocab_size),                shift_labels.view(-1),                ignore_index=-100,            )        return {"loss": loss, "logits": logits}

In [ ]:
%%writefile src/tokenizer.pyimport osfrom typing import Iterable, Iterator, Optionalfrom datasets import load_datasetfrom tokenizers import Tokenizer, decoders, models, normalizers, pre_tokenizers, trainersfrom transformers import AutoTokenizer, PreTrainedTokenizerFastSPECIAL_TOKENS = ["<|endoftext|>", "<|pad|>", "<|bos|>", "<|unk|>"]def _stream_corpus_texts(sources: list[dict], max_chars: int) -> Iterator[str]:    total_chars = 0    for src in sources:        if total_chars >= max_chars:            break        try:            ds = load_dataset(                src["dataset_id"],                name=src.get("config_name"),                data_dir=src.get("data_dir"),                split=src.get("split", "train"),                streaming=True,            )            field = src.get("text_field", "text")            for row in ds:                text = row.get(field)                if isinstance(text, str) and len(text.strip()) > 0:                    yield text                    total_chars += len(text)                    if total_chars >= max_chars:                        break        except Exception as exc:            print(f"[tokenizer] Warning: could not stream {src.get('name', src['dataset_id'])}: {exc}")def _read_local_jsonl(path: str) -> Iterator[str]:    import json    with open(path, 'r', encoding='utf-8') as f:        for line in f:            line = line.strip()            if not line:                continue            try:                obj = json.loads(line)                text = obj.get('text') or obj.get('content') or obj.get('data')                if isinstance(text, str) and text.strip():                    yield text            except json.JSONDecodeError:                continuedef _read_local_parquet(path: str) -> Iterator[str]:    try:        import pyarrow.parquet as pq        table = pq.read_table(path)        for col in ['text', 'content', 'data']:            if col in table.column_names:                for val in table[col].to_pylist():                    if isinstance(val, str) and val.strip():                        yield val                break    except Exception:        passdef _read_local_corpus(data_dir: str, max_chars: int) -> Iterator[str]:    total_chars = 0    for fname in sorted(os.listdir(data_dir)):        if total_chars >= max_chars:            break        fpath = os.path.join(data_dir, fname)        if fname.endswith('.jsonl'):            for text in _read_local_jsonl(fpath):                yield text                total_chars += len(text)                if total_chars >= max_chars:                    break        elif fname.endswith('.parquet'):            for text in _read_local_parquet(fpath):                yield text                total_chars += len(text)                if total_chars >= max_chars:                    breakdef train_bpe_tokenizer(    texts: Iterable[str],    vocab_size: int,    out_dir: str,) -> PreTrainedTokenizerFast:    os.makedirs(out_dir, exist_ok=True)    tokenizer = Tokenizer(models.BPE(unk_token="<|unk|>"))    tokenizer.normalizer = normalizers.NFKC()    tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)    tokenizer.decoder = decoders.ByteLevel()    trainer = trainers.BpeTrainer(        vocab_size=vocab_size,        min_frequency=2,        special_tokens=SPECIAL_TOKENS,        show_progress=True,    )    tokenizer.train_from_iterator(texts, trainer=trainer)    hf_tok = PreTrainedTokenizerFast(        tokenizer_object=tokenizer,        unk_token="<|unk|>",        pad_token="<|pad|>",        bos_token="<|bos|>",        eos_token="<|endoftext|>",    )    hf_tok.save_pretrained(out_dir)    return hf_tokdef load_or_build_tokenizer(tok_cfg: dict, data_sources: list[dict]):    explicit_path = tok_cfg.get("pretrained_or_path")    if explicit_path and os.path.exists(explicit_path):        tok = AutoTokenizer.from_pretrained(explicit_path)        if tok.pad_token_id is None:            tok.pad_token = tok.eos_token or "<|endoftext|>"        return tok    out_dir = tok_cfg.get("tokenizer_out_dir", "/kaggle/working/tokenizer_65k")    if os.path.exists(os.path.join(out_dir, "tokenizer.json")):        tok = PreTrainedTokenizerFast.from_pretrained(out_dir)        if tok.pad_token_id is None:            tok.pad_token = "<|pad|>"        return tok    if tok_cfg.get("train_if_missing", False):        max_chars = int(tok_cfg.get("sample_chars_for_training", 20_000_000))        vocab_size = int(tok_cfg.get("vocab_size", 65536))                # Try local data first (offline mode)        local_data_dir = tok_cfg.get("local_data_dir")        if local_data_dir and os.path.exists(local_data_dir):            print(f"[tokenizer] Training on local data from {local_data_dir}")            corpus = _read_local_corpus(local_data_dir, max_chars)        else:            print("[tokenizer] Training on streamed HF datasets")            corpus = _stream_corpus_texts(data_sources, max_chars=max_chars)                try:            return train_bpe_tokenizer(corpus, vocab_size=vocab_size, out_dir=out_dir)        except Exception as exc:            print(f"[tokenizer] Training failed: {exc}")    # Fallback: try loading a local pretrained tokenizer (e.g., from input dataset)    fallback_name = tok_cfg.get("fallback_tokenizer", "gpt2")    local_fallback = tok_cfg.get("local_fallback_path")    if local_fallback and os.path.exists(local_fallback):        print(f"[tokenizer] Loading local fallback from {local_fallback}")        tok = AutoTokenizer.from_pretrained(local_fallback)    else:        print(f"[tokenizer] Loading pretrained fallback: {fallback_name}")        tok = AutoTokenizer.from_pretrained(fallback_name)        if tok.pad_token_id is None:        tok.pad_token = tok.eos_token    return tok

In [ ]:
%%writefile src/data.pyimport randomimport jsonimport osfrom typing import Iterator, Optionalfrom datasets import load_datasetimport torchfrom torch.utils.data import IterableDatasetdef _normalize_weights(sources: list[dict]) -> list[float]:    raw = [max(0.0, float(s.get("weight", 1.0))) for s in sources]    total = sum(raw)    if total <= 0:        return [1.0 / len(sources) for _ in sources]    return [w / total for w in raw]def _read_local_jsonl(path: str) -> Iterator[str]:    with open(path, 'r', encoding='utf-8') as f:        for line in f:            line = line.strip()            if not line:                continue            try:                obj = json.loads(line)                text = obj.get('text') or obj.get('content') or obj.get('data')                if isinstance(text, str) and text.strip():                    yield text            except json.JSONDecodeError:                continuedef _read_local_parquet(path: str) -> Iterator[str]:    try:        import pyarrow.parquet as pq        table = pq.read_table(path)        for col in ['text', 'content', 'data']:            if col in table.column_names:                for val in table[col].to_pylist():                    if isinstance(val, str) and val.strip():                        yield val                break    except Exception:        passdef _stream_local_corpus(data_dir: str, source_idx: int, seed: int,                           rank: int, world_size: int,                           worker_id: int, num_workers: int,                          shuffle_buffer: int) -> Iterator[str]:    """Stream local JSONL/Parquet files with sharding."""    files = sorted([f for f in os.listdir(data_dir)                    if f.endswith('.jsonl') or f.endswith('.parquet')])    if not files:        return        global_shard_id = rank * num_workers + worker_id    total_shards = world_size * num_workers    rng = random.Random(seed + source_idx * 997 + global_shard_id)        doc_idx = 0    for fname in files:        fpath = os.path.join(data_dir, fname)        if fname.endswith('.jsonl'):            reader = _read_local_jsonl(fpath)        elif fname.endswith('.parquet'):            reader = _read_local_parquet(fpath)        else:            continue                for text in reader:            if doc_idx % total_shards == global_shard_id:                yield text            doc_idx += 1class PackedMixtureDataset(IterableDataset):    """    Streams multiple text/code/math corpora according to domain weights,    tokenizes documents with EOS delimiters, and packs them into fixed-length    `(input_ids, labels)` chunks of length `seq_len`.    Supports both streaming from HF Hub and local JSONL/Parquet files (offline).    """    def __init__(        self,        sources: list[dict],        tokenizer,        seq_len: int = 2048,        seed: int = 42,        shuffle_buffer: int = 10000,        rank: int = 0,        world_size: int = 1,    ) -> None:        super().__init__()        self.sources = sources        self.weights = _normalize_weights(sources)        self.tokenizer = tokenizer        self.seq_len = seq_len        self.seed = seed        self.shuffle_buffer = shuffle_buffer        self.rank = rank        self.world_size = world_size        tokenizer_eos = tokenizer.eos_token_id if tokenizer.eos_token_id is not None else 0        self.default_eos_id = tokenizer_eos    def _make_source_iterator(self, src: dict, source_idx: int):        worker_info = torch.utils.data.get_worker_info()        worker_id = worker_info.id if worker_info is not None else 0        num_workers = worker_info.num_workers if worker_info is not None else 1        global_shard_id = self.rank * num_workers + worker_id        total_shards = self.world_size * num_workers        rng = random.Random(self.seed + source_idx * 997 + global_shard_id)        # Synthetic dataset for smoke tests        if src.get("dataset_id") == "synthetic":            vocab_size = getattr(self.tokenizer, 'vocab_size', 1024)            valid_tokens = list(range(1, min(vocab_size, 1024)))            doc_idx = 0            while True:                doc_len = rng.randint(20, 100)                tokens = [rng.choice(valid_tokens) for _ in range(doc_len)]                if doc_idx % total_shards == global_shard_id:                    yield tokens                doc_idx += 1                if not src.get("loop", True) and doc_idx >= 100:                    break            return        # Local data directory (offline mode)        local_dir = src.get("local_dir")        if local_dir and os.path.exists(local_dir):            yield from _stream_local_corpus(                local_dir, source_idx, self.seed,                 self.rank, self.world_size, worker_id, num_workers, self.shuffle_buffer            )            return        # Streaming from HF Hub (online mode)        ds = load_dataset(            src["dataset_id"],            name=src.get("config_name"),            data_dir=src.get("data_dir"),            split=src.get("split", "train"),            streaming=bool(src.get("streaming", True)),        )        if hasattr(ds, "shuffle") and self.shuffle_buffer > 0:            ds = ds.shuffle(seed=self.seed + source_idx, buffer_size=self.shuffle_buffer)        field = src.get("text_field", "text")        doc_idx = 0        while True:            for row in ds:                if doc_idx % total_shards == global_shard_id:                    text = row.get(field)                    if isinstance(text, str) and text.strip():                        yield text                doc_idx += 1            if not src.get("loop", True):                break    def __iter__(self) -> Iterator[dict[str, torch.Tensor]]:        worker_info = torch.utils.data.get_worker_info()        worker_id = worker_info.id if worker_info is not None else 0        rng = random.Random(self.seed + self.rank * 10007 + worker_id * 131)        iterators = []        active_weights = []        is_synthetic_list = []        eos_ids = []        for idx, src in enumerate(self.sources):            try:                it = self._make_source_iterator(src, idx)                iterators.append(it)                active_weights.append(self.weights[idx])                is_synthetic = src.get("dataset_id") == "synthetic"                is_synthetic_list.append(is_synthetic)                if is_synthetic:                    eos_ids.append(1)                else:                    eos_ids.append(self.default_eos_id)            except Exception as exc:                print(f"[data] Skipping source {src.get('name', src['dataset_id'])}: {exc}")        if not iterators:            raise RuntimeError("No data sources could be initialized.")        w_sum = sum(active_weights)        norm_w = [w / w_sum for w in active_weights]        token_buffer: list[int] = []        while True:            chosen_idx = rng.choices(range(len(iterators)), weights=norm_w, k=1)[0]            chosen_it = iterators[chosen_idx]            is_synthetic = is_synthetic_list[chosen_idx]            try:                item = next(chosen_it)            except StopIteration:                continue            if is_synthetic:                ids = item            else:                text = item                ids = self.tokenizer.encode(text, add_special_tokens=False)            if not ids:                continue            token_buffer.extend(ids)            token_buffer.append(eos_ids[chosen_idx])            while len(token_buffer) >= self.seq_len:                chunk = token_buffer[: self.seq_len]                token_buffer = token_buffer[self.seq_len :]                x = torch.tensor(chunk, dtype=torch.long)                yield {"input_ids": x, "labels": x.clone()}

In [ ]:
%%writefile src/eval.pyimport mathfrom typing import Iterableimport torchimport torch.nn.functional as Ffrom src.model import Llama7BForCausalLM@torch.no_grad()def generate_greedy(    model: Llama7BForCausalLM,    tokenizer,    prompt: str,    device: torch.device,    max_new_tokens: int = 48,) -> str:    was_training = model.training    model.eval()    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)    max_len = model.config.max_position_embeddings    for _ in range(max_new_tokens):        ctx_ids = input_ids[:, -max_len:]        out = model(input_ids=ctx_ids, labels=None)        next_token = torch.argmax(out["logits"][:, -1, :], dim=-1, keepdim=True)        input_ids = torch.cat([input_ids, next_token], dim=-1)        if tokenizer.eos_token_id is not None and next_token.item() == tokenizer.eos_token_id:            break    text = tokenizer.decode(input_ids[0], skip_special_tokens=True)    if was_training:        model.train()    return text@torch.no_grad()def evaluate_model(    model: Llama7BForCausalLM,    tokenizer,    loader: Iterable,    device: torch.device,    precision: str = "bf16",    eval_batches: int = 20,    prompts: list[str] | None = None,) -> dict:    was_training = model.training    model.eval()    total_loss = 0.0    count = 0    loader_iter = iter(loader)    for _ in range(eval_batches):        try:            batch = next(loader_iter)        except StopIteration:            break        input_ids = batch["input_ids"].to(device, non_blocking=True)        labels = batch["labels"].to(device, non_blocking=True)        amp_enabled = device.type == "cuda" and precision in ("bf16", "fp16")        amp_dtype = torch.bfloat16 if (precision == "bf16" and torch.cuda.is_bf16_supported()) else torch.float16        with torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=amp_enabled):            out = model(input_ids=input_ids, labels=labels)            loss = out["loss"]        if loss is not None and torch.isfinite(loss):            total_loss += float(loss.item())            count += 1    mean_loss = total_loss / max(1, count)    perplexity = math.exp(min(mean_loss, 20.0))    generations = []    for p in (prompts or []):        full_out = generate_greedy(model, tokenizer, p, device=device, max_new_tokens=40)        generations.append({"prompt": p, "completion": full_out})    if was_training:        model.train()    return {        "val_loss": mean_loss,        "perplexity": perplexity,        "generations": generations,    }

In [ ]:
%%writefile src/train.pyimport argparseimport globimport jsonimport mathimport osimport timefrom typing import Optionalimport torchimport torch.distributed as distfrom torch.nn.parallel import DistributedDataParallel as DDPfrom torch.utils.data import DataLoaderimport yamlfrom src.data import PackedMixtureDatasetfrom src.eval import evaluate_modelfrom src.model import Llama7BForCausalLM, ModelConfigfrom src.tokenizer import load_or_build_tokenizerdef setup_distributed() -> tuple[int, int, int, torch.device]:    if "RANK" in os.environ and "WORLD_SIZE" in os.environ:        rank = int(os.environ["RANK"])        world_size = int(os.environ["WORLD_SIZE"])        local_rank = int(os.environ.get("LOCAL_RANK", 0))        backend = "nccl" if torch.cuda.is_available() else "gloo"        dist.init_process_group(backend=backend)        if torch.cuda.is_available():            torch.cuda.set_device(local_rank)            device = torch.device(f"cuda:{local_rank}")        else:            device = torch.device("cpu")        return rank, world_size, local_rank, device    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")    return 0, 1, 0, devicedef cleanup_distributed(world_size: int) -> None:    if world_size > 1 and dist.is_initialized():        dist.destroy_process_group()def get_lr(step: int, warmup_steps: int, max_steps: int, max_lr: float, min_lr_ratio: float) -> float:    min_lr = max_lr * min_lr_ratio    if step < warmup_steps:        return max_lr * float(step + 1) / float(max(1, warmup_steps))    if step >= max_steps:        return min_lr    progress = float(step - warmup_steps) / float(max(1, max_steps - warmup_steps))    cosine = 0.5 * (1.0 + math.cos(math.pi * progress))    return min_lr + (max_lr - min_lr) * cosinedef get_amp_context(precision: str, device: torch.device):    if device.type != "cuda" or precision == "fp32":        return torch.autocast(device_type=device.type, enabled=False)    if precision == "bf16":        if torch.cuda.is_bf16_supported():            return torch.autocast(device_type="cuda", dtype=torch.bfloat16)        precision = "fp16"    return torch.autocast(device_type="cuda", dtype=torch.float16)def find_latest_checkpoint(output_dir: str) -> Optional[str]:    ckpt_dirs = sorted(glob.glob(os.path.join(output_dir, "step_*")))    if not ckpt_dirs:        return None    ckpt_dirs.sort(key=lambda p: int(os.path.basename(p).split("_")[-1]))    return ckpt_dirs[-1]def save_checkpoint(    output_dir: str,    step: int,    model_raw: Llama7BForCausalLM,    optimizer: torch.optim.Optimizer,    scaler: torch.cuda.amp.GradScaler,    tokens_seen: int,    max_to_keep: int,) -> str:    ckpt_dir = os.path.join(output_dir, f"step_{step:07d}")    os.makedirs(ckpt_dir, exist_ok=True)    torch.save(model_raw.state_dict(), os.path.join(ckpt_dir, "model.pt"))    torch.save(optimizer.state_dict(), os.path.join(ckpt_dir, "optimizer.pt"))    if scaler.is_enabled():        torch.save(scaler.state_dict(), os.path.join(ckpt_dir, "scaler.pt"))    with open(os.path.join(ckpt_dir, "trainer_state.json"), "w", encoding="utf-8") as f:        json.dump({"step": step, "tokens_seen": tokens_seen}, f, indent=2)    all_ckpts = sorted(        glob.glob(os.path.join(output_dir, "step_*")),        key=lambda p: int(os.path.basename(p).split("_")[-1]),    )    if len(all_ckpts) > max_to_keep:        for old_dir in all_ckpts[:-max_to_keep]:            for fname in os.listdir(old_dir):                os.remove(os.path.join(old_dir, fname))            os.rmdir(old_dir)    return ckpt_dirdef load_checkpoint(    ckpt_dir: str,    model_raw: Llama7BForCausalLM,    optimizer: torch.optim.Optimizer,    scaler: torch.cuda.amp.GradScaler,    device: torch.device,) -> tuple[int, int]:    model_state = torch.load(os.path.join(ckpt_dir, "model.pt"), map_location=device)    model_raw.load_state_dict(model_state)    opt_path = os.path.join(ckpt_dir, "optimizer.pt")    if os.path.exists(opt_path):        optimizer.load_state_dict(torch.load(opt_path, map_location=device))    scaler_path = os.path.join(ckpt_dir, "scaler.pt")    if scaler.is_enabled() and os.path.exists(scaler_path):        scaler.load_state_dict(torch.load(scaler_path, map_location=device))    state_path = os.path.join(ckpt_dir, "trainer_state.json")    step, tokens_seen = 0, 0    if os.path.exists(state_path):        with open(state_path, "r", encoding="utf-8") as f:            st = json.load(f)            step = int(st.get("step", 0))            tokens_seen = int(st.get("tokens_seen", 0))    return step, tokens_seendef apply_debug_smoke_overrides(cfg: dict) -> dict:    cfg["model"]["vocab_size"] = 1024    cfg["model"]["hidden_size"] = 128    cfg["model"]["intermediate_size"] = 256    cfg["model"]["num_hidden_layers"] = 2    cfg["model"]["num_attention_heads"] = 4    cfg["model"]["num_key_value_heads"] = 2    cfg["model"]["max_position_embeddings"] = 256    cfg["model"]["gradient_checkpointing"] = False    cfg["tokenizer"]["train_if_missing"] = False    cfg["tokenizer"]["fallback_tokenizer"] = "gpt2"    cfg["data"]["seq_len"] = 64    cfg["data"]["num_workers"] = 0    cfg["data"]["shuffle_buffer"] = 10    # Use synthetic data for smoke test to avoid network    cfg["data"]["sources"] = [        {            "name": "synthetic",            "dataset_id": "synthetic",            "config_name": None,            "data_dir": None,            "split": "train",            "text_field": "text",            "weight": 1.0,            "streaming": False,        }    ]    cfg["training"]["micro_batch_size"] = 2    cfg["training"]["gradient_accumulation_steps"] = 1    cfg["training"]["max_steps"] = 4    cfg["training"]["warmup_steps"] = 1    cfg["training"]["log_interval"] = 1    cfg["training"]["save_interval"] = 2    cfg["training"]["eval_interval"] = 2    cfg["eval"]["eval_batches"] = 2    cfg["eval"]["completion_prompts"] = []    return cfgdef main() -> None:    parser = argparse.ArgumentParser(description="7B LLM Pretraining Runner")    parser.add_argument("--config", type=str, required=True, help="Path to YAML config")    parser.add_argument("--debug-smoke", action="store_true", help="Run tiny smoke test")    args = parser.parse_args()    with open(args.config, "r", encoding="utf-8") as f:        cfg = yaml.safe_load(f)    if args.debug_smoke:        cfg = apply_debug_smoke_overrides(cfg)    rank, world_size, _, device = setup_distributed()    is_main = rank == 0    seed = int(cfg.get("seed", 42)) + rank    torch.manual_seed(seed)    if torch.cuda.is_available():        torch.cuda.manual_seed_all(seed)    if cfg.get("cache_dir"):        os.environ.setdefault("HF_DATASETS_CACHE", cfg["cache_dir"])    model_cfg = ModelConfig(**cfg["model"])    tokenizer = load_or_build_tokenizer(cfg["tokenizer"], cfg["data"]["sources"])    if not args.debug_smoke and hasattr(tokenizer, "vocab_size") and tokenizer.vocab_size:        if int(cfg["model"].get("vocab_size", tokenizer.vocab_size)) != int(tokenizer.vocab_size):            model_cfg.vocab_size = int(tokenizer.vocab_size)    model = Llama7BForCausalLM(model_cfg).to(device)    if is_main:        counts = model.count_parameters()        print(            f"[model] Params -> total={counts['total']:,} ({counts['total']/1e9:.2f}B), "            f"non_embed={counts['non_embedding']:,} ({counts['non_embedding']/1e9:.2f}B)"        )    if world_size > 1:        ddp_kwargs = {"device_ids": [device.index]} if device.type == "cuda" else {}        model = DDP(model, **ddp_kwargs)    model_raw = model.module if isinstance(model, DDP) else model    train_cfg = cfg["training"]    opt = torch.optim.AdamW(        model_raw.parameters(),        lr=float(train_cfg["learning_rate"]),        betas=(float(train_cfg["adam_beta1"]), float(train_cfg["adam_beta2"])),        eps=float(train_cfg["adam_eps"]),        weight_decay=float(train_cfg["weight_decay"]),    )    precision = train_cfg.get("precision", "bf16")    use_scaler = (device.type == "cuda") and (precision == "fp16") and (not torch.cuda.is_bf16_supported())    scaler = torch.cuda.amp.GradScaler(enabled=use_scaler)    output_dir = cfg["output_dir"]    os.makedirs(output_dir, exist_ok=True)    step = 0    tokens_seen = 0    resume_mode = str(train_cfg.get("resume_from_checkpoint", "auto"))    ckpt_to_load = None    if resume_mode in ("auto", "latest"):        ckpt_to_load = find_latest_checkpoint(output_dir)    elif resume_mode != "none" and os.path.exists(resume_mode):        ckpt_to_load = resume_mode    if ckpt_to_load:        step, tokens_seen = load_checkpoint(ckpt_to_load, model_raw, optimizer=opt, scaler=scaler, device=device)        if is_main:            print(f"[resume] Loaded checkpoint {ckpt_to_load} at step={step}, tokens_seen={tokens_seen:,}")    dataset = PackedMixtureDataset(        sources=cfg["data"]["sources"],        tokenizer=tokenizer,        seq_len=int(cfg["data"]["seq_len"]),        seed=int(cfg.get("seed", 42)),        shuffle_buffer=int(cfg["data"].get("shuffle_buffer", 10000)),        rank=rank,        world_size=world_size,    )    loader = DataLoader(        dataset,        batch_size=int(train_cfg["micro_batch_size"]),        num_workers=int(cfg["data"].get("num_workers", 2)),        pin_memory=(device.type == "cuda"),    )    data_iter = iter(loader)    grad_accum = int(train_cfg["gradient_accumulation_steps"])    max_steps = int(train_cfg["max_steps"])    warmup_steps = int(train_cfg["warmup_steps"])    max_lr = float(train_cfg["learning_rate"])    min_lr_ratio = float(train_cfg.get("min_lr_ratio", 0.1))    max_grad_norm = float(train_cfg.get("max_grad_norm", 1.0))    log_interval = int(train_cfg.get("log_interval", 10))    save_interval = int(train_cfg.get("save_interval", 1000))    eval_interval = int(train_cfg.get("eval_interval", 1000))    max_ckpts = int(train_cfg.get("max_checkpoints_to_keep", 3))    max_runtime_sec = float(train_cfg.get("max_runtime_seconds", 39600))    start_time = time.time()    model.train()    while step < max_steps:        elapsed = time.time() - start_time        if elapsed >= max_runtime_sec:            if is_main:                print(f"[time-guard] Reached {elapsed:.0f}s budget; saving checkpoint before exit.")                save_checkpoint(output_dir, step, model_raw, opt, scaler, tokens_seen, max_ckpts)            break        lr = get_lr(step, warmup_steps, max_steps, max_lr, min_lr_ratio)        for pg in opt.param_groups:            pg["lr"] = lr        opt.zero_grad(set_to_none=True)        accum_loss = 0.0        step_tokens = 0        for micro_idx in range(grad_accum):            batch = next(data_iter)            input_ids = batch["input_ids"].to(device, non_blocking=True)            labels = batch["labels"].to(device, non_blocking=True)            step_tokens += input_ids.numel() * world_size            sync_ctx = (                model.no_sync()                if (isinstance(model, DDP) and micro_idx < grad_accum - 1)                else torch.autocast(device_type=device.type, enabled=False)            )            with sync_ctx:                with get_amp_context(precision, device):                    out = model(input_ids=input_ids, labels=labels)                    loss = out["loss"] / grad_accum                if scaler.is_enabled():                    scaler.scale(loss).backward()                else:                    loss.backward()            accum_loss += float(loss.detach().item())        if scaler.is_enabled():            scaler.unscale_(opt)        grad_norm = torch.nn.utils.clip_grad_norm_(model_raw.parameters(), max_grad_norm)        if scaler.is_enabled():            scaler.step(opt)            scaler.update()        else:            opt.step()        step += 1        tokens_seen += step_tokens        if is_main and (step % log_interval == 0 or step == 1):            dt = max(time.time() - start_time, 1e-6)            tok_per_sec = tokens_seen / dt            print(                f"step={step}/{max_steps} | loss={accum_loss:.4f} | lr={lr:.2e} | "                f"grad_norm={float(grad_norm):.3f} | tokens={tokens_seen:,} | avg_tok/s={tok_per_sec:,.0f}"            )        if eval_interval > 0 and (step % eval_interval == 0 or step == max_steps):            eval_res = evaluate_model(                model=model_raw,                tokenizer=tokenizer,                loader=loader,                device=device,                precision=precision,                eval_batches=int(cfg["eval"].get("eval_batches", 10)),                prompts=cfg["eval"].get("completion_prompts", []),            )            if is_main:                print(                    f"[eval @ step {step}] val_loss={eval_res['val_loss']:.4f} | "                    f"ppl={eval_res['perplexity']:.2f}"                )                for sample in eval_res["generations"]:                    print(f"  > Prompt: {sample['prompt']}\n    Out: {sample['completion']}")            model.train()        if is_main and save_interval > 0 and (step % save_interval == 0 or step == max_steps):            ckpt_path = save_checkpoint(                output_dir=output_dir,                step=step,                model_raw=model_raw,                optimizer=opt,                scaler=scaler,                tokens_seen=tokens_seen,                max_to_keep=max_ckpts,            )            print(f"[checkpoint] Saved {ckpt_path}")    cleanup_distributed(world_size)if __name__ == "__main__":    main()

In [ ]:
%%writefile scripts/launch_kaggle.sh#!/usr/bin/env bashset -euo pipefailCONFIG_PATH="${1:-configs/pretrain_7b_kaggle.yaml}"MODE="${2:-train}"export PYTHONPATH="${PYTHONPATH:-}:$(pwd)"export TOKENIZERS_PARALLELISM="false"if [[ "$MODE" == "smoke" ]]; then  python3 -m src.train --config "$CONFIG_PATH" --debug-smoke  exit 0fiNUM_GPUS=$(python3 -c "import torch; print(torch.cuda.device_count() if torch.cuda.is_available() else 0)")echo "Detected GPUs: $NUM_GPUS"if [[ "$NUM_GPUS" -gt 1 ]]; then  torchrun --standalone --nproc_per_node="$NUM_GPUS" -m src.train --config "$CONFIG_PATH"else  python3 -m src.train --config "$CONFIG_PATH"fi

In [ ]:
# Cell: Smoke test (2 min, tiny model to verify everything works)!chmod +x scripts/launch_kaggle.sh!python -m src.train --config configs/pretrain_7b_kaggle.yaml --debug-smoke

In [ ]:
# Cell: Full 7B training (run after smoke test passes)!bash scripts/launch_kaggle.sh configs/pretrain_7b_kaggle.yaml train